<a href="https://colab.research.google.com/github/Sameekshaingole/fraud-detection-federated-learning/blob/main/notebooks/Streamlit_ui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install streamlit pyngrok -q

print("Streamlit and ngrok installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 104.0 MB/s eta 0:00:00
Streamlit and ngrok installed!


In [3]:
from pyngrok import ngrok

ngrok.set_auth_token("3BnqLvHeZ3gddavL9xtNePRzixY_5BufFg3xV4vxMjXW7Gru5")

print("ngrok configured!")

ngrok configured!


In [15]:
import pickle

raw_models = pickle.load(open('/content/models/federated_model.pkl', 'rb'))
final_model = raw_models[0]
pickle.dump(final_model, open('/content/models/federated_model.pkl', 'wb'))

# Verify
check = pickle.load(open('/content/models/federated_model.pkl', 'rb'))
print(type(check))  # Should print: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
print("✅ Model fixed!")

<class 'sklearn.tree._classes.DecisionTreeClassifier'>
✅ Model fixed!


In [43]:
%%writefile /content/app.py

import streamlit as st
import numpy as np
import pickle
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# ─── PAGE CONFIG ───
st.set_page_config(
    page_title="FraudShield — AI Fraud Detection",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# ─── CUSTOM CSS ───
st.markdown("""
<style>
.main { background-color: #f8f9fc; }
.hero-title { font-size: 2.8rem; font-weight: 800; color: #1a1a2e; text-align: center; margin-bottom: 0; }
.hero-subtitle { font-size: 1.1rem; color: #666; text-align: center; margin-top: 4px; }
.metric-card { background: white; border-radius: 16px; padding: 20px; text-align: center; border: 1px solid #e8ecf4; box-shadow: 0 2px 8px rgba(0,0,0,0.06); }
.metric-value { font-size: 2rem; font-weight: 700; color: #185FA5; }
.metric-label { font-size: 0.85rem; color: #888; margin-top: 4px; }
.section-header { font-size: 1.3rem; font-weight: 700; color: #1a1a2e; border-left: 4px solid #185FA5; padding-left: 12px; margin: 16px 0 12px 0; }
.info-card { background: white; border-radius: 12px; padding: 16px; border: 1px solid #e8ecf4; margin: 8px 0; }
.stTabs [data-baseweb="tab-list"] { gap: 8px; background: white; border-radius: 12px; padding: 6px; border: 1px solid #e8ecf4; }
.stTabs [data-baseweb="tab"] { border-radius: 8px; padding: 8px 20px; font-weight: 600; }
#MainMenu {visibility: hidden;}
footer {visibility: hidden;}
</style>
""", unsafe_allow_html=True)

# ─── LOAD ALL RESOURCES ───
@st.cache_resource
def load_all():
    fed_models = pickle.load(open('/content/models/federated_model.pkl', 'rb'))
    scaler     = pickle.load(open('/content/models/scaler.pkl', 'rb'))
    features   = pickle.load(open('/content/models/feature_columns.pkl', 'rb'))
    medians    = pickle.load(open('/content/models/feature_medians.pkl', 'rb'))
    threshold  = pickle.load(open('/content/models/federated_threshold.pkl', 'rb'))

    # Always ensure it's a list of models
    if not isinstance(fed_models, list):
        fed_models = [fed_models]

    return fed_models, scaler, features, medians, threshold

# ─── INITIALIZE ───
model_loaded = False
fed_models, scaler, feature_columns, feature_medians, fed_threshold = None, None, None, None, 0.5

try:
    fed_models, scaler, feature_columns, feature_medians, fed_threshold = load_all()
    model_loaded = True
except Exception as e:
    st.error(f"❌ Model loading error: {e}")
    st.stop()

# ─── FEDERATED PREDICTION FUNCTION ───
def federated_predict(fed_models, input_vector, threshold=0.5):
    all_probs = []
    for model in fed_models:
        probs = model.predict_proba(input_vector)[:, 1]
        all_probs.append(probs)
    avg_prob = np.mean(all_probs, axis=0)
    prediction = (avg_prob >= threshold).astype(int)
    return prediction[0], avg_prob[0]

# ─── BUILD INPUT VECTOR ───
def build_input_vector(user_inputs, feature_columns, feature_medians):
    input_vector = {col: feature_medians.get(col, 0) for col in feature_columns}

    card4_map   = {'Visa': 0, 'Mastercard': 1, 'American Express': 2, 'Discover': 3}
    card6_map   = {'Debit': 0, 'Credit': 1}
    product_map = {'W': 0, 'H': 1, 'C': 2, 'S': 3, 'R': 4}
    email_map   = {'gmail.com': 0, 'yahoo.com': 1, 'hotmail.com': 2, 'outlook.com': 3, 'other': 4}
    match_map   = {'Yes': 1, 'No': 0}

    if 'TransactionAmt' in input_vector: input_vector['TransactionAmt'] = user_inputs['amount']
    if 'card4'          in input_vector: input_vector['card4']          = card4_map.get(user_inputs['card_network'], 0)
    if 'card6'          in input_vector: input_vector['card6']          = card6_map.get(user_inputs['card_type'], 0)
    if 'ProductCD'      in input_vector: input_vector['ProductCD']      = product_map.get(user_inputs['product'], 0)
    if 'P_emaildomain'  in input_vector: input_vector['P_emaildomain']  = email_map.get(user_inputs['email_domain'], 4)
    if 'TransactionDT'  in input_vector: input_vector['TransactionDT']  = user_inputs['hour'] * 3600
    if 'M1'             in input_vector: input_vector['M1']             = match_map.get(user_inputs['name_match'], 1)
    if 'M2'             in input_vector: input_vector['M2']             = match_map.get(user_inputs['addr_match'], 1)
    if 'M3'             in input_vector: input_vector['M3']             = match_map.get(user_inputs['zip_match'], 1)
    if 'dist1'          in input_vector: input_vector['dist1']          = user_inputs['distance']

    return np.array([input_vector[col] for col in feature_columns]).reshape(1, -1)

# ════════════════════════════════════════
# HERO HEADER
# ════════════════════════════════════════
st.markdown('<h1 class="hero-title">🛡️ FraudShield</h1>', unsafe_allow_html=True)
st.markdown('<p class="hero-subtitle">AI-powered fraud detection using Federated Learning + Homomorphic Encryption</p>', unsafe_allow_html=True)
st.markdown("")

# ─── TOP STATS (your real values) ───
c1, c2, c3, c4, c5 = st.columns(5)
for col, val, label in [
    (c1, "3",       "Banks Connected"),
    (c2, "5",       "Training Rounds"),
    (c3, "0.88",    "ROC-AUC Score"),
    (c4, "~55%",    "Fraud Recall"),
    (c5, "128-bit", "Encryption"),
]:
    with col:
        st.markdown(f"""
        <div class="metric-card">
            <div class="metric-value">{val}</div>
            <div class="metric-label">{label}</div>
        </div>""", unsafe_allow_html=True)

st.markdown("---")

# ════════════════════════════════════════
# TABS
# ════════════════════════════════════════
tab1, tab2, tab3 = st.tabs([
    "🔍 Fraud Detector",
    "📊 Model Dashboard",
    "🔐 Privacy & Security"
])

# ════════════════════════════════════════
# TAB 1 — FRAUD DETECTOR
# ════════════════════════════════════════
with tab1:
    st.markdown('<div class="section-header">Transaction Fraud Detector</div>', unsafe_allow_html=True)
    st.caption(f"Fill in transaction details below. Prediction uses ensemble of {len(fed_models)} bank models with threshold = {fed_threshold}.")
    st.markdown("")

    with st.form("fraud_form"):
        st.markdown("#### Enter Transaction Details")

        row1_c1, row1_c2, row1_c3 = st.columns(3)
        row2_c1, row2_c2, row2_c3 = st.columns(3)
        row3_c1, row3_c2, row3_c3 = st.columns(3)
        row4_c1, _ = st.columns([1, 2])

        with row1_c1:
            amount = st.number_input("💰 Transaction Amount (₹)", min_value=0.0, max_value=1000000.0, value=5000.0, step=500.0)
        with row1_c2:
            card_network = st.selectbox("💳 Card Network", ["Visa", "Mastercard", "American Express", "Discover"])
        with row1_c3:
            card_type = st.selectbox("🏦 Card Type", ["Debit", "Credit"])
        with row2_c1:
            product = st.selectbox("🛍️ Product Type", ["W", "H", "C", "S", "R"], help="W=Web, H=Home, C=Cash, S=Services, R=Retail")
        with row2_c2:
            email_domain = st.selectbox("📧 Buyer Email Domain", ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "other"])
        with row2_c3:
            hour = st.slider("🕐 Transaction Hour", min_value=0, max_value=23, value=14)
        with row3_c1:
            name_match = st.radio("👤 Name on Card Matches?", ["Yes", "No"], horizontal=True)
        with row3_c2:
            addr_match = st.radio("🏠 Address Matches?", ["Yes", "No"], horizontal=True)
        with row3_c3:
            zip_match = st.radio("📮 Zip Code Matches?", ["Yes", "No"], horizontal=True)
        with row4_c1:
            distance = st.number_input("📍 Distance Between Addresses (km)", min_value=0.0, max_value=5000.0, value=0.0, step=10.0)

        st.markdown("")
        submitted = st.form_submit_button("🔍 Analyze Transaction", type="primary", use_container_width=True)

    if submitted:
        user_inputs = {
            'amount': amount, 'card_network': card_network, 'card_type': card_type,
            'product': product, 'email_domain': email_domain, 'hour': hour,
            'name_match': name_match, 'addr_match': addr_match,
            'zip_match': zip_match, 'distance': distance,
        }

        with st.spinner(f"Analyzing transaction across all {len(fed_models)} bank models..."):
            try:
                input_vector = build_input_vector(user_inputs, feature_columns, feature_medians)
                pred, prob   = federated_predict(fed_models, input_vector, fed_threshold)

                st.markdown("---")
                st.markdown("### Prediction Result")
                res_col1, res_col2 = st.columns([1, 1])

                with res_col1:
                    if pred == 1:
                        st.markdown("""
                        <div style="background:linear-gradient(135deg,#ff6b6b,#ee5a24);color:white;
                        padding:24px;border-radius:16px;text-align:center;font-size:1.6rem;font-weight:800;">
                        🚨 FRAUD DETECTED!
                        </div>""", unsafe_allow_html=True)
                        st.error("This transaction has been flagged for review!")
                    else:
                        st.markdown("""
                        <div style="background:linear-gradient(135deg,#00b894,#00cec9);color:white;
                        padding:24px;border-radius:16px;text-align:center;font-size:1.6rem;font-weight:800;">
                        ✅ TRANSACTION LEGITIMATE
                        </div>""", unsafe_allow_html=True)
                        st.success("Transaction appears safe to process!")

                    st.markdown("")
                    m1, m2, m3 = st.columns(3)
                    with m1: st.metric("Fraud Probability", f"{prob*100:.1f}%")
                    with m2: st.metric("Model Decision",    "FRAUD" if pred == 1 else "SAFE")
                    with m3: st.metric("Banks Consulted",   str(len(fed_models)))

                    st.markdown("")
                    st.markdown("**What the model used:**")
                    st.markdown(f"- 💰 Amount: ₹{amount:,.0f}")
                    st.markdown(f"- 💳 Card: {card_network} {card_type}")
                    st.markdown(f"- 🕐 Hour: {hour}:00")
                    st.markdown(f"- 📮 Zip Match: {zip_match}")
                    st.markdown(f"- 👤 Name Match: {name_match}")
                    st.markdown(f"- 📍 Distance: {distance} km")
                    st.markdown(f"- 🤖 Auto-filled: {len(feature_columns) - 10} features filled with training medians")
                    st.markdown(f"- 🎯 Threshold used: {fed_threshold}")

                with res_col2:
                    gauge_color = "#E24B4A" if prob > fed_threshold else "#1D9E75"
                    fig = go.Figure(go.Indicator(
                        mode="gauge+number+delta",
                        value=round(prob * 100, 1),
                        domain={'x': [0, 1], 'y': [0, 1]},
                        title={'text': "Fraud Risk Score", 'font': {'size': 18}},
                        delta={'reference': fed_threshold * 100,
                               'increasing': {'color': "#E24B4A"},
                               'decreasing': {'color': "#1D9E75"}},
                        gauge={
                            'axis'       : {'range': [0, 100], 'tickwidth': 1},
                            'bar'        : {'color': gauge_color, 'thickness': 0.3},
                            'bgcolor'    : "white",
                            'borderwidth': 2,
                            'steps'      : [
                                {'range': [0,  30],  'color': '#e8f5e9'},
                                {'range': [30, 60],  'color': '#fff3e0'},
                                {'range': [60, 100], 'color': '#ffebee'},
                            ],
                            'threshold'  : {
                                'line'     : {'color': "red", 'width': 4},
                                'thickness': 0.75,
                                'value'    : fed_threshold * 100
                            }
                        }
                    ))
                    fig.update_layout(
                        height=320, margin=dict(t=60, b=20, l=20, r=20),
                        paper_bgcolor='white', font={'color': "#1a1a2e"}
                    )
                    st.plotly_chart(fig, use_container_width=True)

                st.caption("🔐 Prediction made by federated ensemble of 3 bank models using Federated Learning + Homomorphic Encryption — raw data never shared!")

            except Exception as e:
                st.error(f"❌ Prediction error: {e}")

# ════════════════════════════════════════
# TAB 2 — MODEL DASHBOARD
# ════════════════════════════════════════
with tab2:
    st.markdown('<div class="section-header">Model Performance Dashboard</div>', unsafe_allow_html=True)

    mc1, mc2, mc3 = st.columns(3)

    # ✅ UPDATED MODEL INFO (same UI + new values + accuracy added)
    models_info = [
        {"name": "Centralized Baseline", "col": mc1,
         "accuracy": 0.9673,
         "precision": 0.6006, "recall": 0.1950, "f1": 0.2944, "roc": 0.8186,
         "privacy": "NONE", "color": "#E24B4A", "badge": "🔴",
         "desc": "Simple Random Forest — all data centralized", "time": "63.4 sec"},

        {"name": "Federated Learning", "col": mc2,
         "accuracy": 0.9492,
         "precision": 0.3526, "recall": 0.5420, "f1": 0.4273, "roc": 0.8744,
         "privacy": "HIGH", "color": "#378ADD", "badge": "🟡",
         "desc": "Random Forest FedAvg — 3 banks, 5 rounds", "time": "173 sec"},

        {"name": "Federated + HE", "col": mc3,
         "accuracy": 0.9492,
         "precision": 0.3526, "recall": 0.5420, "f1": 0.4273, "roc": 0.8744,
         "privacy": "MAXIMUM", "color": "#1D9E75", "badge": "🟢",
         "desc": "Federated + TenSEAL CKKS 128-bit", "time": "180 sec"},
    ]

    # ✅ CARD UI (with accuracy)
    for m in models_info:
        with m['col']:
            st.markdown(f"""
            <div class="info-card" style="border-top:4px solid {m['color']};min-height:300px;">
                <b style="font-size:1rem;color:#1a1a2e;">{m['name']}</b><br>
                <small style="color:#888;">{m['desc']}</small>
                <hr style="margin:10px 0;">
                <table style="width:100%;font-size:0.95rem;">
                    <tr><td style="color:#666;">Accuracy</td>  <td style="text-align:right;"><b>{m['accuracy']}</b></td></tr>
                    <tr><td style="color:#666;">Precision</td> <td style="text-align:right;"><b>{m['precision']}</b></td></tr>
                    <tr><td style="color:#666;">Recall</td>    <td style="text-align:right;"><b>{m['recall']}</b></td></tr>
                    <tr><td style="color:#666;">F1 Score</td>  <td style="text-align:right;"><b>{m['f1']}</b></td></tr>
                    <tr><td style="color:#666;">ROC-AUC</td>   <td style="text-align:right;"><b>{m['roc']}</b></td></tr>
                    <tr><td style="color:#666;">Train Time</td><td style="text-align:right;"><b>{m['time']}</b></td></tr>
                </table><br>
                <span style="background:{m['color']}22;color:{m['color']};padding:4px 12px;
                border-radius:20px;font-size:0.85rem;font-weight:700;">
                    {m['badge']} Privacy: {m['privacy']}
                </span>
            </div>""", unsafe_allow_html=True)

    st.markdown("")
    ch1, ch2 = st.columns(2)

    # ✅ UPDATED BAR CHART (MATCHES IMAGE 1)
    with ch1:
        st.markdown("**All Metrics Comparison**")

        metrics   = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

        baseline  = [0.9673, 0.6006, 0.1950, 0.2944, 0.8186]
        federated = [0.9492, 0.3526, 0.5420, 0.4273, 0.8744]
        fed_he    = [0.9492, 0.3526, 0.5420, 0.4273, 0.8744]

        fig_bar = go.Figure()

        for name, vals, color in [
            ("Centralized Baseline", baseline, "#E24B4A"),
            ("Federated Learning", federated, "#378ADD"),
            ("Federated + HE", fed_he, "#1D9E75"),
        ]:
            fig_bar.add_trace(go.Bar(
                name=name,
                x=metrics,
                y=vals,
                marker_color=color,
                opacity=0.85,
                text=[f'{v:.3f}' for v in vals],
                textposition='outside'
            ))

        fig_bar.update_layout(
            barmode='group',
            height=340,
            margin=dict(t=30,b=20,l=10,r=10),
            legend=dict(orientation='h', yanchor='bottom', y=1.02),
            plot_bgcolor='white',
            paper_bgcolor='white',
            yaxis=dict(gridcolor='#f0f0f0', range=[0, 1.2])
        )

        st.plotly_chart(fig_bar, use_container_width=True)

    # ✅ CONFUSION MATRIX (REPLACES LINE CHART — MATCHES IMAGE 2)
    with ch2:
        import seaborn as sns
        import matplotlib.pyplot as plt
        import numpy as np

        st.markdown("**Centralized Baseline Confusion Matrix (Threshold = 0.5)**")

        cm = np.array([
            [113439, 536],
            [3327,   806]
        ])

        fig, ax = plt.subplots()

        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'],
            ax=ax
        )

        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

        st.pyplot(fig)

    # ✅ KEY FINDINGS (same)
    st.markdown('<div class="section-header">Key Findings</div>', unsafe_allow_html=True)

    f1c, f2c, f3c, f4c = st.columns(4)

    for col, icon, val, label, color in [
        (f1c, "🎯", "+66%", "More fraud caught vs baseline", "#185FA5"),
        (f2c, "📊", "5%", "ROC-AUC gap vs centralized", "#1D9E75"),
        (f3c, "🔐", "0%", "Performance cost of HE", "#0F6E56"),
        (f4c, "⚡", "63%", "Faster than centralized", "#534AB7"),
    ]:
        with col:
            st.markdown(f"""
            <div class="metric-card">
                <div style="font-size:1.8rem;">{icon}</div>
                <div style="font-size:2rem;font-weight:700;color:{color};">{val}</div>
                <div class="metric-label">{label}</div>
            </div>""", unsafe_allow_html=True)

# ════════════════════════════════════════
# TAB 3 — PRIVACY & SECURITY
# ════════════════════════════════════════
with tab3:
    st.markdown('<div class="section-header">Privacy & Security Dashboard</div>', unsafe_allow_html=True)
    p1, p2 = st.columns([1, 1], gap="large")

    with p1:
        st.markdown("#### 🏦 Bank Network Status")
        for bank, txns, fraud, color in [
            ("Bank A", "157,478", "5,531", "#1D9E75"),
            ("Bank B", "157,477", "5,565", "#185FA5"),
            ("Bank C", "157,477", "5,434", "#534AB7"),
        ]:
            st.markdown(f"""
            <div class="info-card" style="border-left:4px solid {color};margin-bottom:12px;">
                <div style="display:flex;justify-content:space-between;align-items:center;">
                    <b style="color:{color};font-size:1rem;">🏦 {bank}</b>
                    <span style="background:#e8f5e9;color:#2e7d32;padding:3px 10px;
                    border-radius:20px;font-size:0.8rem;font-weight:600;">● Active</span>
                </div>
                <hr style="margin:8px 0;">
                <table style="width:100%;font-size:0.9rem;">
                    <tr><td style="color:#666;">Transactions</td>  <td style="text-align:right;"><b>{txns}</b></td></tr>
                    <tr><td style="color:#666;">Fraud cases</td>   <td style="text-align:right;"><b>{fraud}</b></td></tr>
                    <tr><td style="color:#666;">Raw data shared</td><td style="text-align:right;"><b style="color:#1D9E75;">None ✅</b></td></tr>
                    <tr><td style="color:#666;">Training</td>      <td style="text-align:right;"><b>Local only ✅</b></td></tr>
                </table>
            </div>""", unsafe_allow_html=True)

    with p2:
        st.markdown("#### 🔐 Encryption Status")
        for label, value, color in [
            ("🔐 Homomorphic Encryption", "ACTIVE",    "#1D9E75"),
            ("🏦 Raw Data Sharing",        "NONE",      "#1D9E75"),
            ("⚡ Model Weights",           "ENCRYPTED", "#1D9E75"),
            ("🛡️ Security Level",         "128-BIT",   "#185FA5"),
            ("📋 Scheme Used",             "CKKS",      "#185FA5"),
            ("⏱️ Encryption Time",        "0.02 SEC",  "#534AB7"),
            ("🧮 Features Encrypted",      str(len(feature_columns) if feature_columns else 224), "#534AB7"),
        ]:
            st.markdown(f"""
            <div style="display:flex;justify-content:space-between;align-items:center;
            padding:10px 16px;background:white;border-radius:10px;
            border:1px solid #e8ecf4;margin:6px 0;">
                <span style="font-weight:500;font-size:0.95rem;">{label}</span>
                <span style="background:{color}22;color:{color};padding:3px 12px;
                border-radius:20px;font-size:0.85rem;font-weight:700;">{value}</span>
            </div>""", unsafe_allow_html=True)

        st.markdown("")
        st.markdown("""
        <div class="info-card">
            <b>🔒 Level 1 — Federated Learning</b><br>
            <small style="color:#666;">Raw customer data never leaves any bank. Each bank trains locally and only shares model outputs.</small>
            <br><br>
            <b>🔐 Level 2 — Homomorphic Encryption</b><br>
            <small style="color:#666;">Model weights encrypted with TenSEAL CKKS before sharing. Server aggregates without reading values. Only the bank with the private key can decrypt.</small>
        </div>""", unsafe_allow_html=True)

    st.markdown("---")
    st.markdown('<div class="section-header">How It Works</div>', unsafe_allow_html=True)
    step_cols = st.columns(6)
    steps = [
        ("1", "#185FA5", "Local Training",  "Each bank trains on its own private data"),
        ("2", "#534AB7", "Encrypt Weights", "Weights encrypted with TenSEAL CKKS"),
        ("3", "#0F6E56", "Share Encrypted", "Only ciphertext sent to server"),
        ("4", "#854F0B", "FedAvg",          "Server averages encrypted weights"),
        ("5", "#993C1D", "Decrypt",         "Bank decrypts with private key"),
        ("6", "#1D9E75", "Global Model",    "Updated model deployed to all banks"),
    ]
    for col, (num, color, title, desc) in zip(step_cols, steps):
        with col:
            st.markdown(f"""
            <div style="text-align:center;padding:14px 8px;background:white;
            border-radius:12px;border:1px solid #e8ecf4;min-height:150px;">
                <div style="width:38px;height:38px;border-radius:50%;background:{color};
                color:white;font-weight:800;font-size:1.1rem;display:flex;
                align-items:center;justify-content:center;margin:0 auto 10px;">{num}</div>
                <b style="font-size:0.85rem;color:#1a1a2e;">{title}</b><br><br>
                <small style="color:#888;font-size:0.78rem;">{desc}</small>
            </div>""", unsafe_allow_html=True)

Overwriting /content/app.py


In [44]:
import shutil, os

# Copy threshold file too
shutil.copy2(
    '/content/drive/MyDrive/fraud_detection_project/federated_threshold.pkl',
    '/content/models/federated_threshold.pkl'
)

# Also re-copy federated model (as original list — app handles it now)
shutil.copy2(
    '/content/drive/MyDrive/fraud_detection_project/federated_model.pkl',
    '/content/models/federated_model.pkl'
)

print("✅ Files copied:", os.listdir('/content/models'))

✅ Files copied: ['scaler.pkl', 'feature_columns.pkl', 'federated_threshold.pkl', 'feature_medians.pkl', 'federated_model.pkl']


In [45]:
import subprocess, threading, time
from pyngrok import ngrok

ngrok.kill()

def run():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ])

thread = threading.Thread(target=run, daemon=True)
thread.start()
time.sleep(6)

public_url = ngrok.connect(8501)
print("=" * 50)
print(f"🛡️  FraudShield LIVE at: {public_url}")
print("=" * 50)

🛡️  FraudShield LIVE at: NgrokTunnel: "https://shieldlessly-downy-zoila.ngrok-free.dev" -> "http://localhost:8501"
